# LangChain: Q&A over Documents

An example might be a tool that would allow you to query a product catalog for items of interest.

In [13]:
# pip install -U langchain langchain-classic langchain-anthropic langchain-community sentence-transformers

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

Note: LLM's do not always produce the same results. When executing the code in your notebook, you may get slightly different answers that those in the video.

This notebook uses **Anthropic Claude Haiku** for generation (`ChatAnthropic`) and **Hugging Face** embeddings locally (no OpenAI API key for embeddings). Set `ANTHROPIC_API_KEY` in your `.env` (same pattern as `prompts/langchain_models_prompts_output_parsers.ipynb`). The course video uses OpenAI; concepts are the same.

**LangChain v1:** legacy chains like `RetrievalQA` live in the **`langchain-classic`** package (`pip install langchain-classic`), not `langchain.chains`.

> **If you see `NotFoundError` / 404 for `model:`:** update `llm_model` to a current Haiku id from [Anthropic’s models page](https://docs.anthropic.com/en/docs/about-claude/models).

In [ ]:
# Claude Haiku — same default as other notebooks in this repo
llm_model = "claude-haiku-4-5-20251001"

In [ ]:
from langchain_classic.chains import RetrievalQA
from langchain_anthropic import ChatAnthropic
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import DocArrayInMemorySearch
from IPython.display import display, Markdown
from langchain_community.embeddings import HuggingFaceEmbeddings

# Local embeddings (Anthropic does not provide text-embedding APIs)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

/Users/trentoncreamer/Crypto/DataCore Labs/Training/LangChain/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
file = 'OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file)

In [19]:
from langchain_classic.indexes import VectorstoreIndexCreator

In [20]:
#pip install docarray

In [21]:
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=embeddings,
).from_loaders([loader])

In [23]:
query ="Please list all your shirts with sun protection \
in a table in markdown and summarize each one."

**Note**:
- The original course used older LangChain + OpenAI completions for `index.query()`. Here we use **Claude Haiku** for that query step.
- **Embeddings** use Hugging Face `all-MiniLM-L6-v2` locally (no OpenAI embedding API).
- Answers may differ from the video; the flow is the same.

In [24]:
llm_replacement_model = ChatAnthropic(
    temperature=0,
    model=llm_model,
    max_tokens=4096,
)

response = index.query(query, llm=llm_replacement_model)

In [25]:
display(Markdown(response))

# Sun Protection Shirts

| Product ID | Name | Type | UPF Rating | Key Features |
|---|---|---|---|---|
| 679 | Women's Tropical Tee, Sleeveless | Sleeveless button-up | UPF 50+ | Slightly fitted, SunSmart™ protection, wrinkle resistant, low-profile pockets, front and back cape venting, blocks 98% UV rays |
| 255 | Sun Shield Shirt | Short-sleeve | UPF 50+ | Slightly fitted, moisture-wicking, quick-drying, abrasion resistant, fits over swimsuit, recommended by Skin Cancer Foundation, blocks 98% UV rays |
| 619 | Tropical Breeze Shirt | Long-sleeve (men's) | UPF 50+ | Traditional fit, wrinkle-resistant, moisture-wicking, lightweight and breathable, originally designed for fishing, blocks 98% UV rays |
| 374 | Men's Plaid Tropic Shirt, Short-Sleeve | Short-sleeve (men's) | UPF 50+ | Ultracomfortable, wrinkle-free, quick-drying, front and back cape venting, two front bellows pockets, originally designed for fishing, blocks 98% UV rays |

## Summary

All four shirts offer the highest-rated sun protection available at **UPF 50+**, blocking 98% of the sun's harmful UV rays. They feature high-performance fabrics with moisture-wicking and wrinkle-resistant properties. Options include a women's sleeveless style and three men's styles (long-sleeve and short-sleeve variations). Most are designed for outdoor activities like fishing and travel, with practical features such as pockets and venting for comfort in hot weather.

## Step By Step

In [26]:
from langchain_community.document_loaders import CSVLoader
loader = CSVLoader(file_path=file)

In [27]:
docs = loader.load()

In [28]:
docs[0]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \n\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \n\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \n\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \n\nQuestions? Please contact us for any inquiries.")

In [29]:
# Embeddings are defined in the imports cell above (HuggingFaceEmbeddings).
# If you ran this section alone, uncomment:
# from langchain_community.embeddings import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [30]:
embed = embeddings.embed_query("Hi my name is Harrison")

In [31]:
print(len(embed))

384


In [32]:
print(embed[:5])

[-0.06155335530638695, -0.06207892671227455, -0.01895299181342125, 0.048299163579940796, -0.028553981333971024]


In [33]:
db = DocArrayInMemorySearch.from_documents(
    docs, 
    embeddings
)

In [34]:
query = "Please suggest a shirt with sunblocking"

In [35]:
docs = db.similarity_search(query)

In [36]:
len(docs)

4

In [37]:
docs[0]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 255}, page_content=': 255\nname: Sun Shield Shirt by\ndescription: "Block the sun, not the fun – our high-performance sun shirt is guaranteed to protect from harmful UV rays. \n\nSize & Fit: Slightly Fitted: Softly shapes the body. Falls at hip.\n\nFabric & Care: 78% nylon, 22% Lycra Xtra Life fiber. UPF 50+ rated – the highest rated sun protection possible. Handwash, line dry.\n\nAdditional Features: Wicks moisture for quick-drying comfort. Fits comfortably over your favorite swimsuit. Abrasion resistant for season after season of wear. Imported.\n\nSun Protection That Won\'t Wear Off\nOur high-performance fabric provides SPF 50+ sun protection, blocking 98% of the sun\'s harmful rays. This fabric is recommended by The Skin Cancer Foundation as an effective UV protectant.')

In [38]:
retriever = db.as_retriever()

In [39]:
llm = ChatAnthropic(temperature=0.0, model=llm_model, max_tokens=4096)

In [40]:
qdocs = "".join([docs[i].page_content for i in range(len(docs))])


In [41]:
_msg = llm.invoke(
    f"{qdocs} Question: Please list all your \
shirts with sun protection in a table in markdown and summarize each one."
)
response = _msg.content if hasattr(_msg, "content") else str(_msg)


In [42]:
display(Markdown(response))

# Sun Protection Shirts

| Product ID | Name | Type | UPF Rating | Key Features |
|---|---|---|---|---|
| 255 | Sun Shield Shirt | Women's Long-Sleeve | UPF 50+ | 78% nylon, 22% Lycra Xtra Life; Slightly fitted; Moisture-wicking; Abrasion resistant; Handwash, line dry |
| 619 | Tropical Breeze Shirt | Men's Long-Sleeve | UPF 50+ | 71% nylon, 29% polyester; Traditional fit; SunSmart™ technology; Wrinkle-resistant; Cape venting; Machine wash/dry |
| 374 | Men's Plaid Tropic Shirt, Short-Sleeve | Men's Short-Sleeve | UPF 50+ | 52% polyester, 48% nylon; SunSmart technology; Wrinkle-free; Cape venting; Two bellows pockets; Machine wash/dry |
| 679 | Women's Tropical Tee, Sleeveless | Women's Sleeveless | UPF 50+ | 71% nylon, 29% polyester; Slightly fitted; SunSmart™ technology; Wrinkle-resistant; Cape venting; Machine wash/dry |

## Summary

All four shirts offer **UPF 50+ sun protection**, blocking 98% of harmful UV rays and recommended by The Skin Cancer Foundation. They feature moisture-wicking, quick-drying fabrics ideal for outdoor activities. The men's styles (Tropical Breeze and Plaid Tropic) offer traditional or relaxed fits with cape venting for breathability, while the women's styles (Sun Shield and Tropical Tee) provide slightly fitted silhouettes. Care varies—the Sun Shield requires handwashing, while the others are machine washable.

In [ ]:
qa_stuff = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=retriever, 
    verbose=True
)

In [ ]:
query =  "Please list all your shirts with sun protection in a table \
in markdown and summarize each one."

In [ ]:
response = qa_stuff.run(query)

In [ ]:
display(Markdown(response))

In [ ]:
response = index.query(query, llm=llm)

In [ ]:
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=embeddings,
).from_loaders([loader])

Reminder: Download your notebook to you local computer to save your work.